# EfficientNet-B0 - Image-Only Fraud Detection

EfficientNet-B0 for binary fraud classification on the multimodal mixed-all dataset.

**Experiment A - normal**: pos_weight loss, no sampler (baseline)

**Experiment B - sampler**: pos_weight loss + WeightedRandomSampler by combo_type × img_source_dataset

Both experiments use pos_weight because fraud is the majority class in this dataset.

Threshold selection: **minimum expected cost** (FN × $50,000 + FP × $500).


In [1]:
import sys, random, torch
from pathlib import Path
import pandas as pd

sys.path.insert(0, r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src")

from efficientnet_utils import (
    build_model, unfreeze_backbone, fit_model,
    predict_probs, build_train_transform, build_val_transform,
)
from mm_image_utils import load_mm_splits, build_mm_dataloaders, make_loss_fn

def set_seed(seed: int) -> None:
    random.seed(seed); __import__("numpy").random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

SEED = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Random seed: {SEED}")
print(f"Device     : {device}")


Random seed: 42
Device     : cpu


In [2]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
RESULTS_BASE = PROJECT_ROOT / "notebook" / "results" / "mm_image_efficientnet_b0"

TRAIN_CSV = DATA_DIR / "mm_train_mixed_all_group_test.csv"
VAL_CSV   = DATA_DIR / "mm_val_mixed_all_group_test.csv"
TEST_CSV  = DATA_DIR / "mm_test_mixed_all_group_test.csv"

BATCH_SIZE      = 32
NUM_WORKERS     = 0
STAGE1_EPOCHS   = 5;   STAGE1_LR   = 1e-3;  STAGE1_PATIENCE = 5
STAGE2_EPOCHS   = 20;  BACKBONE_LR = 1e-5;  HEAD_LR         = 1e-4
STAGE2_PATIENCE = 6
DROPOUT         = 0.4

EXP_DIRS = {
    "A_normal":  RESULTS_BASE / "A_weighted",   # reuse model trained with pos_weight, no sampler
    "B_sampler": RESULTS_BASE / "B_sampler",    # pos_weight + WeightedRandomSampler — train fresh
}
for d in EXP_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Results base:", RESULTS_BASE)


Project root: C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis
Results base: C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\notebook\results\mm_image_efficientnet_b0


# Image-only baseline


In [3]:
train_df, val_df, test_df = load_mm_splits(TRAIN_CSV, VAL_CSV, TEST_CSV)

# Experiment A — standard shuffle
train_loader, val_loader, test_loader = build_mm_dataloaders(
    train_df, val_df, test_df,
    train_transform=build_train_transform(),
    val_transform=build_val_transform(),
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, use_sampler=False,
)

# Experiment B — WeightedRandomSampler balances batches by combo_type × img_source_dataset
# Prevents MIDV-dominant groups from dominating every batch update.
train_loader_sampler, _, _ = build_mm_dataloaders(
    train_df, val_df, test_df,
    train_transform=build_train_transform(),
    val_transform=build_val_transform(),
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, use_sampler=True,
)

print(f"Batches — train:{len(train_loader)} val:{len(val_loader)} test:{len(test_loader)}")
print("\nTrain class counts:")
print(train_df["final_label"].value_counts())

# Both experiments use pos_weight — fraud is the majority class and needs down-weighting.
LOSS_FNS = {
    "A_normal":  make_loss_fn(train_df, device),
    "B_sampler": make_loss_fn(train_df, device),
}


train: 7,182 rows | label 0:2,588 label 1:4,594 ratio 1:1.78
  combos: {'tab0_img0': 2588, 'tab0_img1': 1866, 'tab1_img0': 863, 'tab1_img1': 1865}
val: 1,508 rows | label 0:557 label 1:951 ratio 1:1.71
  combos: {'tab0_img0': 557, 'tab0_img1': 382, 'tab1_img0': 186, 'tab1_img1': 383}
test: 1,474 rows | label 0:554 label 1:920 ratio 1:1.66
  combos: {'tab0_img0': 554, 'tab0_img1': 368, 'tab1_img0': 185, 'tab1_img1': 367}
WeightedRandomSampler — weighting by: ['combo_type', 'img_source_dataset']
  8 groups:
    tab0_img0_fantasyid: n=484  w=0.002066
    tab0_img0_midv: n=2104  w=0.000475
    tab0_img1_fantasyid: n=858  w=0.001166
    tab0_img1_fmidv: n=1008  w=0.000992
    tab1_img0_fantasyid: n=179  w=0.005587
    tab1_img0_midv: n=684  w=0.001462
    tab1_img1_fantasyid: n=833  w=0.001200
    tab1_img1_fmidv: n=1032  w=0.000969
Batches — train:225 val:48 test:47

Train class counts:
final_label
1    4594
0    2588
Name: count, dtype: int64
  pos_weight = 0.563  (n_neg=2588, n_pos=4594)

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay,
)

FN_COST = 50_000
FP_COST = 500

def evaluate_image_model(y_true, y_prob, split_name="split", threshold=0.5):
    """Mirrors evaluate_model() from tabular utils."""
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "split":     split_name,
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true,    y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true,        y_pred, zero_division=0)),
        "roc_auc":   float(roc_auc_score(y_true, y_prob)),
        "pr_auc":    float(average_precision_score(y_true, y_prob)),
    }
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n=== {split_name.upper()} ===")
    for k, v in metrics.items():
        if k != "split": print(f"{k}: {v:.6f}")
    print("\nConfusion matrix:"); print(cm)
    return metrics, cm

def search_thresholds_image(y_true, y_prob,
                             thresholds=np.arange(0.10, 1.00, 0.05)):
    """Mirrors search_thresholds() from tabular utils."""
    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        rows.append({
            "threshold":       round(float(t), 4),
            "precision":       float(precision_score(y_true, y_pred, zero_division=0)),
            "recall":          float(recall_score(y_true,    y_pred, zero_division=0)),
            "f1":              float(f1_score(y_true,        y_pred, zero_division=0)),
            "predicted_fraud": int(y_pred.sum()),
        })
    return pd.DataFrame(rows)

def compute_cost_table_image(threshold_results, y_val_true,
                              fn_cost=FN_COST, fp_cost=FP_COST):
    """Mirrors compute_cost_table() from tabular utils."""
    n_pos = int((y_val_true == 1).sum())
    n_neg = int((y_val_true == 0).sum())
    rows = []
    for _, row in threshold_results.iterrows():
        tp = round(row["recall"] * n_pos)
        fn = n_pos - tp
        fp = row["predicted_fraud"] - tp
        tn = n_neg - fp
        rows.append({
            "threshold":       row["threshold"],
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision":       row["precision"],
            "recall":          row["recall"],
            "f1":              row["f1"],
            "predicted_fraud": row["predicted_fraud"],
            "expected_cost":   int(fn * fn_cost + fp * fp_cost),
        })
    return pd.DataFrame(rows).sort_values("expected_cost").reset_index(drop=True)

def evaluate_image_with_threshold(y_true, y_prob, threshold):
    """Mirrors evaluate_with_threshold() from tabular utils."""
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    metrics = {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true,    y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true,        y_pred, zero_division=0)),
    }
    print(f"\nTuned threshold: {threshold}")
    print(f"Precision: {metrics['precision']:.6f}")
    print(f"Recall:    {metrics['recall']:.6f}")
    print(f"F1:        {metrics['f1']:.6f}")
    print("\nConfusion matrix:"); print(cm)
    return metrics, cm

def compute_expected_cost_from_cm(cm, fn_cost=FN_COST, fp_cost=FP_COST):
    """Mirrors compute_expected_cost_from_cm() from tabular utils."""
    tn, fp, fn, tp = cm.ravel()
    return {"tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
            "expected_cost": int(fn * fn_cost + fp * fp_cost)}

def save_image_experiment_outputs(results_dir, test_y_true, test_y_prob,
                                   threshold_results, cost_df,
                                   tuned_metrics, tuned_cm,
                                   val_metrics, test_metrics, prefix):
    """Mirrors save_experiment_outputs() from tabular utils."""
    from pathlib import Path
    results_dir = Path(results_dir)
    pred_df = pd.DataFrame({"y_true": test_y_true, "y_prob": test_y_prob})
    pred_df["y_pred"] = (pred_df["y_prob"] >= tuned_metrics["threshold"]).astype(int)
    pred_df.to_csv(results_dir / "test_predictions.csv", index=False)
    threshold_results.to_csv(results_dir / f"{prefix}_threshold_results.csv", index=False)
    cost_df.to_csv(results_dir / f"{prefix}_cost_results.csv", index=False)
    pd.DataFrame([tuned_metrics]).to_csv(results_dir / f"{prefix}_tuned_metrics.csv", index=False)
    pd.DataFrame(tuned_cm, index=["true_0","true_1"],
                 columns=["pred_0","pred_1"]).to_csv(
        results_dir / f"{prefix}_tuned_cm.csv")
    pd.DataFrame([val_metrics, test_metrics]).to_csv(
        results_dir / f"{prefix}_metrics_summary.csv", index=False)

def plot_model_diagnostics_pretty(cm, threshold_results, cost_df, model_name,
                                   fn_cost=FN_COST, fp_cost=FP_COST, save_path=None):
    """Identical to notebooks 11 and 12."""
    cm = np.asarray(cm).astype(int)
    threshold_df   = threshold_results.sort_values("threshold")
    cost_df_sorted = cost_df.sort_values("threshold")
    best_row             = cost_df.loc[cost_df["expected_cost"].idxmin()]
    best_threshold       = float(best_row["threshold"])
    best_validation_cost = float(best_row["expected_cost"])
    tn, fp, fn, tp = cm.ravel()
    test_expected_cost   = fn * fn_cost + fp * fp_cost

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), constrained_layout=True)
    fig.patch.set_facecolor("white")

    ax = axes[0]
    cm_for_color = np.maximum(cm, 1)
    norm = LogNorm(vmin=1, vmax=cm_for_color.max())
    im = ax.imshow(cm_for_color, cmap="Blues", norm=norm)
    ax.set_title("Tuned test confusion matrix", fontsize=12, pad=12)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Legit","Fraud"]); ax.set_yticklabels(["Legit","Fraud"])
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
    for i in range(2):
        for j in range(2):
            value = cm[i,j]; rgba = im.cmap(norm(cm_for_color[i,j]))
            lum = 0.299*rgba[0]+0.587*rgba[1]+0.114*rgba[2]
            ax.text(j, i, f"{value:,}", ha="center", va="center",
                    fontsize=11, fontweight="bold",
                    color="white" if lum < 0.45 else "#08306b")
    for sp in ax.spines.values(): sp.set_visible(False)

    ax = axes[1]
    ax.plot(threshold_df["threshold"], threshold_df["precision"], linewidth=2.2, label="Precision")
    ax.plot(threshold_df["threshold"], threshold_df["recall"],    linewidth=2.2, label="Recall")
    ax.plot(threshold_df["threshold"], threshold_df["f1"],        linewidth=2.2, label="F1")
    ax.axvline(best_threshold, linestyle="--", linewidth=1.8, color="black", alpha=0.7)
    ax.text(0.98, 0.95, f"Best threshold = {best_threshold:.2f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=10,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
    ax.set_title("Validation metrics by threshold", fontsize=12, pad=12)
    ax.set_xlabel("Threshold"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.25); ax.legend(frameon=False, loc="lower left")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[2]
    ax.plot(cost_df_sorted["threshold"], cost_df_sorted["expected_cost"],
            linewidth=2.2, marker="o", markersize=4)
    ax.axvline(best_threshold, linestyle="--", linewidth=1.8, color="black", alpha=0.7)
    ax.scatter(best_threshold, best_validation_cost, s=80, color="black", zorder=5)
    ax.annotate(f"Min cost: ${best_validation_cost/1_000_000:.2f}M",
                xy=(best_threshold, best_validation_cost), xytext=(18,18),
                textcoords="offset points", fontsize=10,
                arrowprops=dict(arrowstyle="->", linewidth=1, color="black"),
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.85))
    ax.set_title("Validation expected cost by threshold", fontsize=12, pad=12)
    ax.set_xlabel("Threshold"); ax.set_ylabel("Expected cost")
    ax.grid(True, alpha=0.25)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1_000_000:.2f}M"))
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle(f"{model_name} diagnostics | Tuned test cost: ${test_expected_cost/1_000_000:.2f}M",
                 fontsize=15, fontweight="bold")
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

print("All helper functions defined.")


All helper functions defined.


## 1. EfficientNet-B0 - A normal

pos_weight loss, no WeightedRandomSampler.
Direct baseline - pos_weight corrects for the majority-fraud class imbalance.



In [5]:
results_dir = EXP_DIRS["A_normal"]
results_dir.mkdir(parents=True, exist_ok=True)

model_path    = results_dir / "stage2_best.pt"
FORCE_RETRAIN = False

print("Checking model path:")
print(model_path)
print("Model exists:", model_path.exists())

if model_path.exists() and not FORCE_RETRAIN:
    print("Already trained. Loading saved EfficientNet-B0 A_normal model...")
    model_normal = build_model(pretrained=False, freeze_backbone=False,
                               dropout=DROPOUT).to(device)
    model_normal.load_state_dict(torch.load(model_path, map_location=device))
    model_normal.eval()
    print("Model loaded successfully.")
else:
    print("Saved model not found. Training EfficientNet-B0 A_normal model...")
    model_normal = build_model(pretrained=True, freeze_backbone=True,
                               dropout=DROPOUT).to(device)
    opt1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_normal.parameters()),
        lr=STAGE1_LR, weight_decay=1e-4)
    model_normal, h_s1 = fit_model(
        model=model_normal, train_loader=train_loader, val_loader=val_loader,
        loss_fn=LOSS_FNS["A_normal"], optimizer=opt1, device=device,
        epochs=STAGE1_EPOCHS, early_stopping_patience=STAGE1_PATIENCE,
        monitor_metric="val_roc_auc",
        model_save_path=results_dir / "stage1_best.pt",
    )
    unfreeze_backbone(model_normal)
    bp = [p for n, p in model_normal.named_parameters() if "classifier" not in n]
    hp = list(model_normal.classifier.parameters())
    opt2 = torch.optim.Adam(
        [{"params": bp, "lr": BACKBONE_LR},
         {"params": hp, "lr": HEAD_LR}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=STAGE2_EPOCHS, eta_min=1e-7)
    h_s1_ref = h_s1 if "h_s1" in dir() else None
    model_normal, h_s2 = fit_model(
        model=model_normal, train_loader=train_loader, val_loader=val_loader,
        loss_fn=LOSS_FNS["A_normal"], optimizer=opt2, scheduler=sched,
        device=device, epochs=STAGE2_EPOCHS,
        early_stopping_patience=STAGE2_PATIENCE,
        monitor_metric="val_roc_auc", freeze_bn=True,
        model_save_path=model_path,
    )
    hist = (pd.concat([h_s1_ref.assign(stage=1), h_s2.assign(stage=2)], ignore_index=True)
            if h_s1_ref is not None else h_s2.copy())
    hist["epoch_global"] = range(1, len(hist)+1)
    hist.to_csv(results_dir / "training_history.csv", index=False)
    print("Training finished and model saved to:", model_path)


Checking model path:
C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\notebook\results\mm_image_efficientnet_b0\A_weighted\stage2_best.pt
Model exists: True
Already trained. Loading saved EfficientNet-B0 A_normal model...
EfficientNet-B0 | image_size=224 | in_features=1280 | dropout=0.4
Model loaded successfully.


In [6]:
val_y_normal,  val_p_normal  = predict_probs(model_normal, val_loader,  device)
test_y_normal, test_p_normal = predict_probs(model_normal, test_loader, device)

val_metrics_normal,  val_cm_normal  = evaluate_image_model(
    val_y_normal,  val_p_normal,  split_name="validation_normal")
test_metrics_normal, test_cm_normal = evaluate_image_model(
    test_y_normal, test_p_normal, split_name="test_normal")



=== VALIDATION_NORMAL ===
accuracy: 0.813660
precision: 0.897862
recall: 0.794953
f1: 0.843279
roc_auc: 0.875052
pr_auc: 0.935144

Confusion matrix:
[[471  86]
 [195 756]]

=== TEST_NORMAL ===
accuracy: 0.827001
precision: 0.901086
recall: 0.811957
f1: 0.854202
roc_auc: 0.878216
pr_auc: 0.936584

Confusion matrix:
[[472  82]
 [173 747]]


The normal EfficientNet-B0 image-only model achieved a test F1-score of 0.8542, with precision of 0.9011 and recall of 0.8120. It detected 747 fraud cases and missed 173, while producing 82 false positives. This is slightly stronger than the normal ResNet-18 image-only baseline, but still weaker than the multimodal models, showing that image information alone is useful but insufficient.

### Threshold tuning for A normal

Cost-based threshold tuning on the validation set:
minimum expected cost (FN × $50,000 + FP × $500).



In [7]:
threshold_results_normal = search_thresholds_image(val_y_normal, val_p_normal)
print(threshold_results_normal)


    threshold  precision    recall        f1  predicted_fraud
0        0.10   0.640110  0.980021  0.774408             1456
1        0.15   0.694253  0.952681  0.803191             1305
2        0.20   0.777574  0.889590  0.829819             1088
3        0.25   0.834853  0.866456  0.850361              987
4        0.30   0.853530  0.851735  0.852632              949
5        0.35   0.868676  0.848580  0.858511              929
6        0.40   0.878855  0.839117  0.858526              908
7        0.45   0.886105  0.818086  0.850738              878
8        0.50   0.897862  0.794953  0.843279              842
9        0.55   0.907731  0.765510  0.830576              802
10       0.60   0.925170  0.715037  0.806643              735
11       0.65   0.934524  0.660358  0.773876              672
12       0.70   0.952537  0.611987  0.745198              611
13       0.75   0.973734  0.545741  0.699461              533
14       0.80   0.989429  0.492114  0.657303              473
15      

In [8]:
cost_df_normal = compute_cost_table_image(threshold_results_normal, val_y_normal)
print(cost_df_normal.head())


   threshold   tp     fp   fn     tn  precision    recall        f1  \
0       0.10  932  524.0   19   33.0   0.640110  0.980021  0.774408   
1       0.15  906  399.0   45  158.0   0.694253  0.952681  0.803191   
2       0.20  846  242.0  105  315.0   0.777574  0.889590  0.829819   
3       0.25  824  163.0  127  394.0   0.834853  0.866456  0.850361   
4       0.30  810  139.0  141  418.0   0.853530  0.851735  0.852632   

   predicted_fraud  expected_cost  
0           1456.0        1212000  
1           1305.0        2449500  
2           1088.0        5371000  
3            987.0        6431500  
4            949.0        7119500  


For the normal EfficientNet-B0 image-only model, the cost-optimal threshold was 0.10. At this threshold, it detected 932 fraud cases on the validation set and missed 19, resulting in an expected validation cost of $1.21M. However, it also produced 524 false positives and correctly identified only 33 legitimate cases. This shows that cost-based tuning makes the image-only EfficientNet-B0 model highly aggressive, while its fraud detection remains weaker than the multimodal models.

In [9]:
val_cost_normal         = compute_expected_cost_from_cm(val_cm_normal)
default_validation_cost = val_cost_normal["expected_cost"]
best_validation_cost    = cost_df_normal.iloc[0]["expected_cost"]
cost_reduction          = default_validation_cost - best_validation_cost

print(f"Default validation cost:    ${default_validation_cost:,.0f}")
print(f"Best tuned validation cost: ${best_validation_cost:,.0f}")
print(f"Cost reduction:             ${cost_reduction:,.0f}")


Default validation cost:    $9,793,000
Best tuned validation cost: $1,212,000
Cost reduction:             $8,581,000


### Tuned threshold results - A normal

Final test performance at the cost-optimal threshold selected on the validation set.

In [10]:
best_threshold_normal = float(cost_df_normal.iloc[0]["threshold"])
best_threshold_normal


0.1

In [11]:
tuned_metrics_normal, tuned_cm_normal = evaluate_image_with_threshold(
    test_y_normal, test_p_normal, best_threshold_normal)
cost_summary_normal = compute_expected_cost_from_cm(tuned_cm_normal)
print(cost_summary_normal)



Tuned threshold: 0.1
Precision: 0.635409
Recall:    0.986957
F1:        0.773095

Confusion matrix:
[[ 33 521]
 [ 12 908]]
{'tp': 908, 'fp': 521, 'fn': 12, 'tn': 33, 'expected_cost': 860500}




After cost-based threshold tuning, the selected threshold for the normal EfficientNet-B0 image-only model was 0.10.

At this threshold, the model detects 908 out of 920 fraud cases and misses 12. It also produces 521 false positives and correctly identifies only 33 legitimate cases.

The expected test cost is:

Expected cost = FN × $50,000 + FP × $500  
Expected cost = 12 × $50,000 + 521 × $500  
Expected cost = $600,000 + $260,500  
Expected cost = $860,500

Compared with the default threshold, cost-based tuning increases fraud recall substantially, but it makes the model very aggressive. The model catches most fraud cases, but legitimate-class detection becomes weak.

Overall, the normal EfficientNet-B0 image-only model performs better than the normal ResNet-18 image-only model in terms of tuned cost.

In [12]:
save_image_experiment_outputs(
    results_dir=results_dir,
    test_y_true=test_y_normal, test_y_prob=test_p_normal,
    threshold_results=threshold_results_normal, cost_df=cost_df_normal,
    tuned_metrics=tuned_metrics_normal, tuned_cm=tuned_cm_normal,
    val_metrics=val_metrics_normal, test_metrics=test_metrics_normal,
    prefix="normal",
)
print("Saved outputs to:", results_dir)


Saved outputs to: C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\notebook\results\mm_image_efficientnet_b0\A_weighted


In [13]:
saved_files = sorted([p.name for p in results_dir.glob("*")])
saved_files


['normal_cost_results.csv',
 'normal_metrics_summary.csv',
 'normal_threshold_results.csv',
 'normal_tuned_cm.csv',
 'normal_tuned_metrics.csv',
 'stage1_best.pt',
 'stage2_best.pt',
 'subgroup_combo_type.csv',
 'test_predictions.csv',
 'threshold_sweep.csv',
 'training_history.csv',
 'weighted_cost_results.csv',
 'weighted_metrics_summary.csv',
 'weighted_threshold_results.csv',
 'weighted_tuned_cm.csv',
 'weighted_tuned_metrics.csv']

## 2. EfficientNet-B0 = B sampler

pos_weight loss + WeightedRandomSampler by combo_type × img_source_dataset.
Balances training batches so MIDV-dominant groups do not dominate every update.



In [ ]:
results_dir = EXP_DIRS["B_sampler"]
results_dir.mkdir(parents=True, exist_ok=True)

model_path    = results_dir / "stage2_best.pt"
FORCE_RETRAIN = False

print("Checking model path:")
print(model_path)
print("Model exists:", model_path.exists())

if model_path.exists() and not FORCE_RETRAIN:
    print("Already trained. Loading saved EfficientNet-B0 B_sampler model...")
    model_sampler = build_model(pretrained=False, freeze_backbone=False,
                                dropout=DROPOUT).to(device)
    model_sampler.load_state_dict(torch.load(model_path, map_location=device))
    model_sampler.eval()
    print("Model loaded successfully.")
else:
    print("Saved model not found. Training EfficientNet-B0 B_sampler model...")
    model_sampler = build_model(pretrained=True, freeze_backbone=True,
                                dropout=DROPOUT).to(device)
    opt1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_sampler.parameters()),
        lr=STAGE1_LR, weight_decay=1e-4)
    model_sampler, h_s1 = fit_model(
        model=model_sampler, train_loader=train_loader_sampler, val_loader=val_loader,
        loss_fn=LOSS_FNS["B_sampler"], optimizer=opt1, device=device,
        epochs=STAGE1_EPOCHS, early_stopping_patience=STAGE1_PATIENCE,
        monitor_metric="val_roc_auc",
        model_save_path=results_dir / "stage1_best.pt",
    )
    unfreeze_backbone(model_sampler)
    bp = [p for n, p in model_sampler.named_parameters() if "classifier" not in n]
    hp = list(model_sampler.classifier.parameters())
    opt2 = torch.optim.Adam(
        [{"params": bp, "lr": BACKBONE_LR},
         {"params": hp, "lr": HEAD_LR}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=STAGE2_EPOCHS, eta_min=1e-7)
    h_s1_ref = h_s1 if "h_s1" in dir() else None
    model_sampler, h_s2 = fit_model(
        model=model_sampler, train_loader=train_loader_sampler, val_loader=val_loader,
        loss_fn=LOSS_FNS["B_sampler"], optimizer=opt2, scheduler=sched,
        device=device, epochs=STAGE2_EPOCHS,
        early_stopping_patience=STAGE2_PATIENCE,
        monitor_metric="val_roc_auc", freeze_bn=True,
        model_save_path=model_path,
    )
    hist = (pd.concat([h_s1_ref.assign(stage=1), h_s2.assign(stage=2)], ignore_index=True)
            if h_s1_ref is not None else h_s2.copy())
    hist["epoch_global"] = range(1, len(hist)+1)
    hist.to_csv(results_dir / "training_history.csv", index=False)
    print("Training finished and model saved to:", model_path)


Checking model path:
C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\notebook\results\mm_image_efficientnet_b0\B_sampler\stage2_best.pt
Model exists: False
Saved model not found. Training EfficientNet-B0 B_sampler model...
EfficientNet-B0 | image_size=224 | in_features=1280 | dropout=0.4
Epoch 01 | lr=1.00e-03 | train_loss=0.3949 | val_loss=0.3907 | val_f1=0.7961 | val_roc_auc=0.8059
  ✓ Saved best model (epoch 1, val_roc_auc=0.8059)
Epoch 02 | lr=1.00e-03 | train_loss=0.3787 | val_loss=0.3672 | val_f1=0.8118 | val_roc_auc=0.8272
  ✓ Saved best model (epoch 2, val_roc_auc=0.8272)
Epoch 03 | lr=1.00e-03 | train_loss=0.3746 | val_loss=0.3707 | val_f1=0.8289 | val_roc_auc=0.8379
  ✓ Saved best model (epoch 3, val_roc_auc=0.8379)
Epoch 04 | lr=1.00e-03 | train_loss=0.3758 | val_loss=0.3655 | val_f1=0.8271 | val_roc_auc=0.8393
  ✓ Saved best model (epoch 4, val_roc_auc=0.8393)
Epoch 05 | lr=1.00e-03 | train_loss=0.3786 | val_loss=0.3688 | val_f1=0.8317 | val_roc_auc=0.8433
 

In [ ]:
val_y_sampler,  val_p_sampler  = predict_probs(model_sampler, val_loader,  device)
test_y_sampler, test_p_sampler = predict_probs(model_sampler, test_loader, device)

val_metrics_sampler,  val_cm_sampler  = evaluate_image_model(
    val_y_sampler,  val_p_sampler,  split_name="validation_sampler")
test_metrics_sampler, test_cm_sampler = evaluate_image_model(
    test_y_sampler, test_p_sampler, split_name="test_sampler")


### Interpretation - B sampler

With the WeightedRandomSampler, batches are balanced across combo_type groups. Comparing with A_normal shows whether batch rebalancing improves detection.

In [ ]:
threshold_results_sampler = search_thresholds_image(val_y_sampler, val_p_sampler)
print(threshold_results_sampler)


In [ ]:
cost_df_sampler = compute_cost_table_image(threshold_results_sampler, val_y_sampler)
print(cost_df_sampler.head())


### Threshold tuning for B sampler

Same cost-based framework as A_normal.

In [ ]:
val_cost_sampler        = compute_expected_cost_from_cm(val_cm_sampler)
default_validation_cost = val_cost_sampler["expected_cost"]
best_validation_cost    = cost_df_sampler.iloc[0]["expected_cost"]
cost_reduction          = default_validation_cost - best_validation_cost

print(f"Default validation cost:    ${default_validation_cost:,.0f}")
print(f"Best tuned validation cost: ${best_validation_cost:,.0f}")
print(f"Cost reduction:             ${cost_reduction:,.0f}")


In [ ]:
best_threshold_sampler = float(cost_df_sampler.iloc[0]["threshold"])
best_threshold_sampler


In [ ]:
tuned_metrics_sampler, tuned_cm_sampler = evaluate_image_with_threshold(
    test_y_sampler, test_p_sampler, best_threshold_sampler)
cost_summary_sampler = compute_expected_cost_from_cm(tuned_cm_sampler)
print(cost_summary_sampler)


### Tuned threshold results - B sampler

In [ ]:
results_dir = EXP_DIRS["B_sampler"]
save_image_experiment_outputs(
    results_dir=results_dir,
    test_y_true=test_y_sampler, test_y_prob=test_p_sampler,
    threshold_results=threshold_results_sampler, cost_df=cost_df_sampler,
    tuned_metrics=tuned_metrics_sampler, tuned_cm=tuned_cm_sampler,
    val_metrics=val_metrics_sampler, test_metrics=test_metrics_sampler,
    prefix="sampler",
)
print("Saved outputs to:", results_dir)


In [ ]:
saved_files = sorted([p.name for p in results_dir.glob("*")])
saved_files


## 3. Comparison - A normal vs B sampler

Both use pos_weight. The varying factor is the WeightedRandomSampler.

In [ ]:
comparison_df = pd.DataFrame([
    {
        "model":               "A_normal",
        "val_f1":              val_metrics_normal["f1"],
        "val_roc_auc":         val_metrics_normal["roc_auc"],
        "val_pr_auc":          val_metrics_normal["pr_auc"],
        "test_f1":             test_metrics_normal["f1"],
        "test_roc_auc":        test_metrics_normal["roc_auc"],
        "test_pr_auc":         test_metrics_normal["pr_auc"],
        "best_threshold":      best_threshold_normal,
        "tuned_test_precision":tuned_metrics_normal["precision"],
        "tuned_test_recall":   tuned_metrics_normal["recall"],
        "tuned_test_f1":       tuned_metrics_normal["f1"],
        "tuned_expected_cost": cost_summary_normal["expected_cost"],
    },
    {
        "model":               "B_sampler",
        "val_f1":              val_metrics_sampler["f1"],
        "val_roc_auc":         val_metrics_sampler["roc_auc"],
        "val_pr_auc":          val_metrics_sampler["pr_auc"],
        "test_f1":             test_metrics_sampler["f1"],
        "test_roc_auc":        test_metrics_sampler["roc_auc"],
        "test_pr_auc":         test_metrics_sampler["pr_auc"],
        "best_threshold":      best_threshold_sampler,
        "tuned_test_precision":tuned_metrics_sampler["precision"],
        "tuned_test_recall":   tuned_metrics_sampler["recall"],
        "tuned_test_f1":       tuned_metrics_sampler["f1"],
        "tuned_expected_cost": cost_summary_sampler["expected_cost"],
    },
])
print(comparison_df)


In [ ]:
comparison_df.sort_values("tuned_expected_cost")[
    ["model","best_threshold","tuned_test_precision",
     "tuned_test_recall","tuned_test_f1","tuned_expected_cost"]
]


The comparison shows whether WeightedRandomSampler improves performance under cost-sensitive evaluation.

## Class imbalance plot

In [ ]:
class_counts = train_df["final_label"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Non-fraud (0)", "Fraud (1)"], class_counts.values)
plt.title("Class distribution in image training data")
plt.ylabel("Number of samples")
plt.show()


## Comparison bar chart for the 2 variants

In [ ]:
import numpy as np

metrics_plot  = ["tuned_test_precision","tuned_test_recall","tuned_test_f1"]
metric_labels = ["Precision","Recall","F1"]
x = np.arange(len(comparison_df["model"])); width = 0.25

plt.figure(figsize=(9, 5))
for i, metric in enumerate(metrics_plot):
    plt.bar(x + i*width, comparison_df[metric], width, label=metric_labels[i])
plt.xticks(x + width, comparison_df["model"])
plt.ylabel("Score")
plt.title("Tuned test performance by model")
plt.legend(); plt.ylim(0, 1); plt.tight_layout(); plt.show()


## Expected cost comparison

In [ ]:
cost_plot_df = comparison_df.sort_values("tuned_expected_cost")
plt.figure(figsize=(8, 5))
bars = plt.bar(cost_plot_df["model"], cost_plot_df["tuned_expected_cost"])
plt.title("Expected cost by image model variant")
plt.ylabel("Expected cost")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1_000_000:.2f}M"))
plt.bar_label(bars,
    labels=[f"${v/1_000_000:.2f}M" for v in cost_plot_df["tuned_expected_cost"]],
    padding=3)
plt.tight_layout(); plt.show()


## Threshold performance plot

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_results_normal["threshold"], threshold_results_normal["precision"], label="Precision")
plt.plot(threshold_results_normal["threshold"], threshold_results_normal["recall"],    label="Recall")
plt.plot(threshold_results_normal["threshold"], threshold_results_normal["f1"],        label="F1")
plt.title("Threshold analysis — A_normal")
plt.xlabel("Threshold"); plt.ylabel("Score"); plt.legend(); plt.show()


## Tuned confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (cm, title_) in zip(axes, [
    (tuned_cm_normal,  "A: Normal"),
    (tuned_cm_sampler, "B: Sampler"),
]):
    ConfusionMatrixDisplay(confusion_matrix=cm,
                           display_labels=["Legit","Fraud"]).plot(
        ax=ax, colorbar=False, values_format="d")
    ax.set_title(title_)
plt.suptitle("Tuned test confusion matrices", y=1.05)
plt.tight_layout(); plt.show()


## Threshold analysis - precision, recall, F1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (df, title_) in zip(axes, [
    (threshold_results_normal,  "A: Normal"),
    (threshold_results_sampler, "B: Sampler"),
]):
    ax.plot(df["threshold"], df["precision"], marker="o", label="Precision")
    ax.plot(df["threshold"], df["recall"],    marker="o", label="Recall")
    ax.plot(df["threshold"], df["f1"],        marker="o", label="F1")
    ax.set_title(title_); ax.set_xlabel("Threshold"); ax.set_ylim(0, 1); ax.grid(True)
axes[0].set_ylabel("Score"); axes[-1].legend(loc="best")
plt.suptitle("Threshold analysis: precision, recall, and F1")
plt.tight_layout(); plt.show()


## Expected cost by threshold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, (df, title_) in zip(axes, [
    (cost_df_normal,  "A: Normal"),
    (cost_df_sampler, "B: Sampler"),
]):
    df_sorted = df.sort_values("threshold")
    best_row  = df.loc[df["expected_cost"].idxmin()]
    ax.plot(df_sorted["threshold"], df_sorted["expected_cost"], marker="o")
    ax.axvline(best_row["threshold"], linestyle="--",
               label=f"Best threshold = {best_row['threshold']:.2f}")
    ax.set_title(title_); ax.set_xlabel("Threshold"); ax.set_ylabel("Expected cost")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1_000_000:.2f}M"))
    ax.grid(True); ax.legend()
plt.suptitle("Expected cost by threshold")
plt.tight_layout(); plt.show()


## Precision-Recall curves

In [ ]:
p_n, r_n, _ = precision_recall_curve(val_y_normal,  val_p_normal)
p_s, r_s, _ = precision_recall_curve(val_y_sampler, val_p_sampler)

plt.figure(figsize=(7, 6))
plt.plot(r_n, p_n, label="A: Normal")
plt.plot(r_s, p_s, label="B: Sampler")
plt.title("Precision-Recall curves on validation set")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend(); plt.show()


In [ ]:
plot_model_diagnostics_pretty(
    cm=tuned_cm_normal,
    threshold_results=threshold_results_normal,
    cost_df=cost_df_normal,
    model_name="EfficientNet-B0 — A normal",
    save_path=RESULTS_BASE / "diagnostics_normal.png",
)


Diagnostics for A_normal: tuned confusion matrix, validation threshold curve, and cost curve.

In [ ]:
plot_model_diagnostics_pretty(
    cm=tuned_cm_sampler,
    threshold_results=threshold_results_sampler,
    cost_df=cost_df_sampler,
    model_name="EfficientNet-B0 — B sampler",
    save_path=RESULTS_BASE / "diagnostics_sampler.png",
)


Diagnostics for B_sampler: tuned confusion matrix, validation threshold curve, and cost curve.

## Final choice

The variant with the lowest expected cost is selected as the final image-only result for comparison with tabular and multimodal models.

In [ ]:
comparison_df.to_csv(RESULTS_BASE / "comparison.csv", index=False)
print("Saved comparison to:", RESULTS_BASE / "comparison.csv")
final_choice_df = comparison_df.sort_values("tuned_expected_cost").copy()
final_choice_df
